Purpose of the prototype space
Compress the multimodal (text+image) embedding into a small set of “sentiment directions.” Each prototype is a representative direction (centroid) for a rating/sentiment class learned from user ratings.

Make prediction simple and interpretable. For a new image–text pair, you compute its embedding once, then:

compare it to each prototype with cosine similarity
pick the closest prototype → predicted sentiment/rating
keep the similarity score as a confidence-like signal.
Enable analysis and visualization. You can inspect:

which ratings form distinct clusters/directions (e.g., 5-star separating well)
which items/users sit near which sentiment direction
where confusion happens (e.g., 4 vs 5) and how “close” points are to competing prototypes.

## Prototype sentiment space from multimodal embeddings\n
\n
This notebook:\n
- Reads `interactions_multimodal.jsonl` (streaming)\n
- Learns **one prototype vector per distinct rating value** (inferred from the dataset; not hardcoded)\n
- Embeds (text,image) pairs into the prototype space and predicts the closest prototype by cosine similarity\n
- Stores predictions + metadata + a saved embedding subset for visualization

In [1]:
from __future__ import annotations

import json
import math
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Tuple

import numpy as np
import pandas as pd

In [2]:
# ---- Paths (portable) ----
# By default, assume the notebook is in the same folder as `interactions_multimodal.jsonl`.
# If you run this notebook elsewhere (e.g., Colab), set `ROOT_OVERRIDE` to the mounted folder.

ROOT_OVERRIDE = None  # e.g. r"/content/drive/MyDrive/DIscussion_14-04-2026" or r"E:\\Meta D\\MMRecSys\\DIscussion_14-04-2026"

ROOT = Path(ROOT_OVERRIDE) if ROOT_OVERRIDE else Path.cwd()
DATA_PATH = ROOT / "interactions_multimodal.jsonl"
ARTIFACT_DIR = ROOT / "artifacts" / "embed_space"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find interactions_multimodal.jsonl at: {DATA_PATH}\n"
        "Set ROOT_OVERRIDE to the directory containing the file."
    )

DATA_PATH, ARTIFACT_DIR

(PosixPath('/content/interactions_multimodal.jsonl'),
 PosixPath('/content/artifacts/embed_space'))

### 1) Streaming reader + schema probe

In [3]:
@dataclass(frozen=True)
class Interaction:
    user_id: str
    asin: str
    parent_asin: Optional[str]
    timestamp: Optional[int]
    rating: float
    text_embedding: np.ndarray
    image_embedding: np.ndarray


def _as_float_array(x: object) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float32)
    if arr.ndim != 1:
        raise ValueError(f"Expected 1D embedding, got shape {arr.shape}")
    return arr


def iter_interactions_jsonl(path: Path) -> Iterator[Interaction]:
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            obj = json.loads(line)

            if "text_embedding" not in obj or "image_embedding" not in obj:
                raise KeyError(
                    f"Line {i}: missing expected keys. Found keys={sorted(obj.keys())}"
                )

            yield Interaction(
                user_id=str(obj.get("user_id", "")),
                asin=str(obj.get("asin", "")),
                parent_asin=(None if obj.get("parent_asin") is None else str(obj.get("parent_asin"))),
                timestamp=(None if obj.get("timestamp") is None else int(obj.get("timestamp"))),
                rating=float(obj.get("rating")),
                text_embedding=_as_float_array(obj["text_embedding"]),
                image_embedding=_as_float_array(obj["image_embedding"]),
            )


def l2_normalize(v: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    v = v.astype(np.float32, copy=False)
    n = float(np.linalg.norm(v))
    if not math.isfinite(n) or n < eps:
        return np.zeros_like(v)
    return v / n


def build_pair_embedding(text_emb: np.ndarray, image_emb: np.ndarray) -> np.ndarray:
    return l2_normalize(np.concatenate([text_emb, image_emb], axis=0))


# Probe first few rows
probe = []
for j, it in enumerate(iter_interactions_jsonl(DATA_PATH)):
    probe.append(
        {
            "user_id": it.user_id,
            "asin": it.asin,
            "rating": it.rating,
            "text_dim": int(it.text_embedding.shape[0]),
            "image_dim": int(it.image_embedding.shape[0]),
        }
    )
    if j >= 4:
        break

pd.DataFrame(probe)

,user_id,asin,rating,text_dim,image_dim
0,AGQMAOHZFQX3JMJIEND3IFN65L5Q,B08CPF22H2,3.0,768,768
1,AGQMAOHZFQX3JMJIEND3IFN65L5Q,B07YXB7TJ4,4.0,768,768
2,AGQMAOHZFQX3JMJIEND3IFN65L5Q,B09B8RZKBJ,3.0,768,768
3,AGQMAOHZFQX3JMJIEND3IFN65L5Q,B0971PWDFB,5.0,768,768
4,AGQMAOHZFQX3JMJIEND3IFN65L5Q,B08XNZYCG4,5.0,768,768


### 2) Learn prototypes (one per distinct rating value, inferred)

In [4]:
# Streaming accumulation: per-rating sum vector and count

rating_to_sum: Dict[float, np.ndarray] = {}
rating_to_count: Dict[float, int] = {}

text_dim = None
image_dim = None
pair_dim = None

rows_meta: List[dict] = []
pair_embeddings: List[np.ndarray] = []

for it in iter_interactions_jsonl(DATA_PATH):
    if text_dim is None:
        text_dim = int(it.text_embedding.shape[0])
        image_dim = int(it.image_embedding.shape[0])
        pair_dim = text_dim + image_dim
    else:
        if int(it.text_embedding.shape[0]) != text_dim:
            raise ValueError(f"Inconsistent text_embedding dim: {it.text_embedding.shape[0]} vs {text_dim}")
        if int(it.image_embedding.shape[0]) != image_dim:
            raise ValueError(f"Inconsistent image_embedding dim: {it.image_embedding.shape[0]} vs {image_dim}")

    pair = build_pair_embedding(it.text_embedding, it.image_embedding).astype(np.float32)
    r = float(it.rating)

    if r not in rating_to_sum:
        rating_to_sum[r] = np.zeros((pair_dim,), dtype=np.float64)
        rating_to_count[r] = 0

    rating_to_sum[r] += pair.astype(np.float64)
    rating_to_count[r] += 1

    rows_meta.append(
        {
            "user_id": it.user_id,
            "asin": it.asin,
            "parent_asin": it.parent_asin,
            "timestamp": it.timestamp,
            "true_rating": r,
        }
    )
    pair_embeddings.append(pair)

distinct_ratings = sorted(rating_to_count.keys())
distinct_ratings, {r: rating_to_count[r] for r in distinct_ratings}

([1.0, 2.0, 3.0, 4.0, 5.0], {1.0: 5, 2.0: 3, 3.0: 17, 4.0: 49, 5.0: 92})

In [5]:
# Build prototypes as normalized means

proto_ratings = np.array(distinct_ratings, dtype=np.float32)
proto_counts = np.array([rating_to_count[r] for r in distinct_ratings], dtype=np.int64)

proto_vectors = []
for r in distinct_ratings:
    mean_vec = (rating_to_sum[r] / max(1, rating_to_count[r])).astype(np.float32)
    proto_vectors.append(l2_normalize(mean_vec))

proto_vectors = np.stack(proto_vectors, axis=0).astype(np.float32)  # [K, D]

proto_vectors.shape, proto_ratings, proto_counts

((5, 1536),
 array([1., 2., 3., 4., 5.], dtype=float32),
 array([ 5,  3, 17, 49, 92]))

### 3) Score pairs against prototypes (cosine similarity) + persist artifacts

In [6]:
E = np.stack(pair_embeddings, axis=0).astype(np.float32)  # [N, D], already normalized
P = proto_vectors.astype(np.float32)  # [K, D], normalized

# Cosine similarity since vectors are L2-normalized
S = E @ P.T  # [N, K]

top_idx = np.argmax(S, axis=1)
pred_rating = proto_ratings[top_idx]
top_sim = S[np.arange(S.shape[0]), top_idx]

pred_df = pd.DataFrame(rows_meta)
pred_df["predicted_rating"] = pred_rating
pred_df["top_cosine_similarity"] = top_sim

# Also store per-prototype cosine scores as columns (visualization-friendly)
for k, r in enumerate(proto_ratings.tolist()):
    pred_df[f"cosine_to_rating_{int(r) if float(r).is_integer() else r}"] = S[:, k]

pred_df.head()

,user_id,asin,parent_asin,timestamp,true_rating,predicted_rating,top_cosine_similarity,cosine_to_rating_1,cosine_to_rating_2,cosine_to_rating_3,cosine_to_rating_4,cosine_to_rating_5
0,AGQMAOHZFQX3JMJIEND3IFN65L5Q,B08CPF22H2,B08CPF22H2,1638278324176,3.0,3.0,0.624760,0.490275,0.414419,0.624760,0.565515,0.521311
1,AGQMAOHZFQX3JMJIEND3IFN65L5Q,B07YXB7TJ4,B07YXB7TJ4,1637557778600,4.0,3.0,0.592841,0.473991,0.395962,0.592841,0.584917,0.521244
2,AGQMAOHZFQX3JMJIEND3IFN65L5Q,B09B8RZKBJ,B09B8RZKBJ,1635762045819,3.0,3.0,0.632738,0.473429,0.405248,0.632738,0.563750,0.497624
3,AGQMAOHZFQX3JMJIEND3IFN65L5Q,B0971PWDFB,B0971PWDFB,1635299757935,5.0,4.0,0.600023,0.442075,0.410963,0.510318,0.600023,0.521376
4,AGQMAOHZFQX3JMJIEND3IFN65L5Q,B08XNZYCG4,B0B6653VGM,1635039959260,5.0,4.0,0.563550,0.484991,0.306084,0.495377,0.563550,0.513012


In [7]:
# Persist artifacts for the visualization notebook

prototypes_path = ARTIFACT_DIR / "prototypes.npz"
predictions_path = ARTIFACT_DIR / "predictions.csv"
embeddings_path = ARTIFACT_DIR / "pair_embeddings.npz"

np.savez_compressed(
    prototypes_path,
    proto_ratings=proto_ratings,
    proto_counts=proto_counts,
    proto_vectors=proto_vectors,
    text_dim=np.array([text_dim], dtype=np.int64),
    image_dim=np.array([image_dim], dtype=np.int64),
)

pred_df.to_csv(predictions_path, index=False)

# Save embeddings + minimal metadata aligned by row index
np.savez_compressed(
    embeddings_path,
    pair_embeddings=E,
    true_rating=pred_df["true_rating"].to_numpy(dtype=np.float32),
    predicted_rating=pred_df["predicted_rating"].to_numpy(dtype=np.float32),
    top_cosine_similarity=pred_df["top_cosine_similarity"].to_numpy(dtype=np.float32),
)

prototypes_path, predictions_path, embeddings_path

(PosixPath('/content/artifacts/embed_space/prototypes.npz'),
 PosixPath('/content/artifacts/embed_space/predictions.csv'),
 PosixPath('/content/artifacts/embed_space/pair_embeddings.npz'))

### 4) Predict sentiment for a new (text,image) pair

Below we demo two options:\n
- Use an existing row as a stand-in “new pair” (quick sanity check)\n
- Or paste your own `text_embedding` and `image_embedding` arrays

In [8]:
def predict_pair(
    text_embedding: np.ndarray,
    image_embedding: np.ndarray,
    proto_vectors: np.ndarray,
    proto_ratings: np.ndarray,
) -> dict:
    pair = build_pair_embedding(text_embedding, image_embedding).astype(np.float32)
    sims = pair @ proto_vectors.T  # [K]
    j = int(np.argmax(sims))
    return {
        "predicted_rating": float(proto_ratings[j]),
        "top_cosine_similarity": float(sims[j]),
        "all_cosine_similarities": {float(proto_ratings[k]): float(sims[k]) for k in range(len(proto_ratings))},
    }


# Demo: treat row 0 as a new pair
demo_it = next(iter_interactions_jsonl(DATA_PATH))
demo_text = demo_it.text_embedding
demo_img = demo_it.image_embedding

predict_pair(demo_text, demo_img, proto_vectors, proto_ratings)

{'predicted_rating': 3.0,
 'top_cosine_similarity': 0.6247600317001343,
 'all_cosine_similarities': {1.0: 0.4902748763561249,
  2.0: 0.4144187569618225,
  3.0: 0.6247600317001343,
  4.0: 0.5655155181884766,
  5.0: 0.5213110446929932}}

In [9]:
# Optional: paste your own embeddings here

# new_text_embedding = np.array([...], dtype=np.float32)
# new_image_embedding = np.array([...], dtype=np.float32)
# predict_pair(new_text_embedding, new_image_embedding, proto_vectors, proto_ratings)
None

### 5) Store a single prediction record (example)

This shows the schema you’ll also see in `predictions.csv`.

In [10]:
# Example: one record from the dataset, plus prediction details
row0 = pred_df.iloc[0].to_dict()
row0

{'user_id': 'AGQMAOHZFQX3JMJIEND3IFN65L5Q',
 'asin': 'B08CPF22H2',
 'parent_asin': 'B08CPF22H2',
 'timestamp': 1638278324176,
 'true_rating': 3.0,
 'predicted_rating': 3.0,
 'top_cosine_similarity': 0.6247601509094238,
 'cosine_to_rating_1': 0.49027496576309204,
 'cosine_to_rating_2': 0.4144187271595001,
 'cosine_to_rating_3': 0.6247601509094238,
 'cosine_to_rating_4': 0.565515398979187,
 'cosine_to_rating_5': 0.5213109254837036}